# BCI Competition IV – Dataset 2b: Exploration and verification

**Master's Thesis — Maestría en Ingeniería, IUE 2026**  
Author: Juan Carlos Guerrero Sierra

---

This notebook verifies that the GDF files are correct, visualises raw and filtered signals, and generates sample ERSP spectrograms for one subject.

**Before starting:**
1. Make sure all 45 GDF files are in `data/raw/` (9 subjects × 5 sessions)
2. Run `uv sync --dev` if you haven't already

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import mne
mne.set_log_level('WARNING')

from config import *
from preprocessing import verify_dataset, load_raw, apply_filter, extract_epochs, plot_subject_overview, plot_all_subjects_summary
from ersp import compute_ersp_image, generate_ersp_for_subject, plot_ersp_examples, plot_ersp_average

print('Modules loaded successfully.')

## 1. Dataset verification

Check that all 45 GDF files are present.

In [ ]:
ok = verify_dataset()
if ok:
    print('\n✓ Dataset complete. Ready to process.')
else:
    print('\n✗ Missing files. Download the dataset from:')
    print('  https://www.bbci.de/competition/iv/download/')

## 2. Inspection of a GDF file

Load subject 1, session 1 (training) and inspect its contents.

In [ ]:
SUBJECT = 1   # change to explore other subjects (1–9)
SESSION = 1   # session number (1–3 training, 4–5 evaluation)

raw = load_raw(SUBJECT, SESSION)

print(f'Subject S{SUBJECT:02d} — Session {SESSION:02d}')
print(f'  Channels:          {raw.ch_names}')
print(f'  Sampling rate:     {raw.info["sfreq"]} Hz')
print(f'  Total duration:    {raw.n_times / raw.info["sfreq"]:.1f} s')
print(f'  Number of samples: {raw.n_times}')

In [ ]:
# Inspect available events in the file
events, event_id = mne.events_from_annotations(raw, verbose=False)
print(f'Events found: {event_id}')
print(f'Total events: {len(events)}')
print(f'\nFirst 10 events:')
for e in events[:10]:
    print(f'  sample={e[0]}, t={e[0]/raw.info["sfreq"]:.2f}s, type={e[2]}')

## 3. Raw vs. filtered signal visualisation

In [ ]:
# Raw signal
data_raw = raw.get_data() * 1e6  # in µV
t = raw.times

# Filtered signal (8–30 Hz)
raw_filt = raw.copy()
apply_filter(raw_filt)
data_filt = raw_filt.get_data() * 1e6

fig, axes = plt.subplots(len(raw.ch_names), 2, figsize=(16, 4*len(raw.ch_names)))
if len(raw.ch_names) == 1:
    axes = axes.reshape(1, -1)

# Show only the first 15 seconds
mask = t < 15

for i, ch_name in enumerate(raw.ch_names):
    axes[i,0].plot(t[mask], data_raw[i, mask], color='#2C7BB6', lw=0.6)
    axes[i,0].set_title(f'{ch_name} — Raw signal')
    axes[i,0].set_ylabel('Amplitude (µV)')
    axes[i,0].spines['top'].set_visible(False)
    axes[i,0].spines['right'].set_visible(False)

    axes[i,1].plot(t[mask], data_filt[i, mask], color='#D7191C', lw=0.6)
    axes[i,1].set_title(f'{ch_name} — Filtered (8–30 Hz)')
    axes[i,1].spines['top'].set_visible(False)
    axes[i,1].spines['right'].set_visible(False)

axes[-1,0].set_xlabel('Time (s)')
axes[-1,1].set_xlabel('Time (s)')
plt.suptitle(f'S{SUBJECT:02d} session {SESSION:02d} — Raw vs. filtered signal (first 15 s)',
             fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Epoch extraction and statistics

In [ ]:
epochs = extract_epochs(raw_filt.copy())

print(f'Epochs extracted:')
print(f'  Total valid:  {len(epochs)}')
print(f'  Left  (0):    {len(epochs["left"]) if "left" in epochs.event_id else 0}')
print(f'  Right (1):    {len(epochs["right"]) if "right" in epochs.event_id else 0}')
print(f'  Window:       {EPOCH_TMIN} to {EPOCH_TMAX} s')
print(f'  Shape per epoch: {epochs.get_data().shape}  (epochs × channels × samples)')

In [ ]:
# Event-related potential (ERP) per class
fig, axes = plt.subplots(1, len(epochs.ch_names), figsize=(5*len(epochs.ch_names), 4))
if len(epochs.ch_names) == 1:
    axes = [axes]

for i, ch in enumerate(epochs.ch_names):
    ax = axes[i]
    for cls, color, label in [('left', '#2C7BB6', 'Left'), ('right', '#D7191C', 'Right')]:
        if cls in epochs.event_id:
            ep = epochs[cls].get_data(picks=[i])[:,0,:] * 1e6
            ax.plot(epochs.times, ep.mean(0), color=color, label=label, lw=1.5)
            ax.fill_between(epochs.times,
                            ep.mean(0)-ep.std(0),
                            ep.mean(0)+ep.std(0),
                            color=color, alpha=0.15)
    ax.axvline(0, color='k', lw=0.8, ls='--', label='Onset')
    ax.axhline(0, color='gray', lw=0.4, ls=':')
    ax.set_title(f'Channel {ch}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('µV' if i==0 else '')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle(f'S{SUBJECT:02d} — Average ERP per class (mean ± SD)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Sample ERSP spectrogram

Compute the ERSP spectrogram for a single trial and visualise it.

In [ ]:
from scipy.signal import stft as scipy_stft

# Take the first trial of each class, channel C3
ch_idx = 0  # C3
bl_end = int(abs(EPOCH_TMIN) * SFREQ)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
cls_list = [('left', 0, 'Left', '#2C7BB6'), ('right', 1, 'Right', '#D7191C')]

for row, (cls_name, cls_label, cls_display, color) in enumerate(cls_list):
    if cls_name not in epochs.event_id:
        continue
    ep_data = epochs[cls_name].get_data()  # (N, C, T)
    signal   = ep_data[0, ch_idx, :]       # first trial, channel C3
    baseline = signal[:bl_end]

    ersp_img = compute_ersp_image(signal, baseline, sfreq=SFREQ)

    # Time-domain signal
    ax = axes[row, 0]
    ax.plot(epochs.times, signal*1e6, color=color, lw=0.8)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.axvspan(EPOCH_TMIN, 0, alpha=0.1, color='gray', label='Baseline')
    ax.set_title(f'EEG signal — {cls_display} (C3, trial 1)')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (µV)')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ERSP spectrogram
    ax = axes[row, 1]
    im = ax.imshow(
        ersp_img, aspect='auto', origin='lower', cmap='RdYlBu_r',
        extent=[EPOCH_TMIN, EPOCH_TMAX, ERSP_FMIN, ERSP_FMAX],
        vmin=0, vmax=1
    )
    ax.axvline(0, color='white', lw=1.0, ls='--')
    ax.axhspan(8, 13, alpha=0.15, color='cyan')    # mu band
    ax.axhspan(14, 30, alpha=0.10, color='yellow') # beta band
    ax.set_title(f'ERSP spectrogram — {cls_display} (C3, trial 1)\n'
                 f'(cyan=mu, yellow=beta)')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    plt.colorbar(im, ax=ax, label='Normalised ERSP [0,1]')

plt.suptitle(f'S{SUBJECT:02d} — EEG signal and ERSP spectrogram per class',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Epoch summary across all subjects

Check how many valid trials exist per subject (useful for identifying problematic subjects).

In [ ]:
# This may take several minutes (loads and processes all 9 subjects)
# Change suffix to 'E' to inspect evaluation sessions
plot_all_subjects_summary(suffix='T')

## 7. Full preprocessing of one subject

If everything looks correct, run the full preprocessing from the terminal:

```bash
# One subject with visualisation
uv run preprocess --subject 1 --plot

# All subjects, both splits
uv run preprocess --subject all --suffix both
```

Then generate the spectrograms:

```bash
uv run ersp --subject all --plot
```

And train the models:

```bash
uv run train --model spectnet --all_subjects
uv run train --model eegnet --all_subjects
uv run train --model shallowconvnet --all_subjects
```

In [ ]:
print('Exploration complete.')
print(f'Figures saved to: {FIGURES_DIR}')